# 1 — Three-Phase VSI Basics

> **Goal.** Understand the 3-phase voltage-source inverter topology
> (6 switches in 3 legs), how SPWM produces a sinusoidal output, and
> introduce the **Clarke** and **Park** transforms that turn
> 3-phase AC into 2-axis DC.

**Prerequisites**

- Half-bridge converter project (`projects/converters/half_bridge/`)
  — a 3-phase VSI is conceptually three half-bridges sharing the
  same DC bus.
- Boost PFC project (`projects/converters/boost_pfc/`) — for the
  AC-side intuition (line voltages, fundamentals, harmonics).

**What you'll know at the end**

1. The 3-phase VSI topology and its **8 switching states** (6 active
   + 2 zero) arranged as a hexagon in the αβ plane.
2. How to generate three SPWM duty signals (one per leg) from three
   reference sinusoids and a triangular carrier — the simplest
   modulation strategy.
3. The **Clarke transform** (abc → αβ): why the $2/3$ factor, why
   amplitude-invariant, what positive-sequence looks like.
4. The **Park transform** (αβ → dq): the rotating-frame insight that
   turns 3-phase AC quantities into DC quantities — the headline
   pedagogical moment of this entire project.


## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from vsi_3phase_model import (
    VSI3PhaseParams,
    clarke_transform, inverse_clarke_transform,
    park_transform, inverse_park_transform,
    spwm_duties,
    simulate_open_loop_svpwm,
    operating_point_report,
    fundamental_rms, thd,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

p = VSI3PhaseParams()
print(operating_point_report(p, mode="standalone"))


## 1. Topology and switching states

A 3-phase VSI has **3 legs**, each with a top switch + bottom switch.
Each leg is a half-bridge sharing the same DC bus:

```
         +V_dc ──┬────┬────┬──── (top switches)
                 │    │    │
                S1   S3   S5
                 │    │    │
                 ├────┼────┼──── output: a, b, c phases
                 │    │    │
                S4   S6   S2
                 │    │    │
            0 ───┴────┴────┴──── (bottom switches)
```

Each leg has two **complementary states** (S_top on, S_bot off — or
vice-versa). With 3 legs, that's $2^3 = 8$ switching states.

In the **αβ plane**, those 8 states map to **6 active vectors**
(forming a hexagon) and **2 zero vectors** (at the origin: all-top
or all-bottom):

```
                  V2 (110)
                  /        \
                 /          \
        V3 (010)            V1 (100)
                |    *V0/V7
        V4 (011)            V6 (101)
                 \          /
                  \        /
                  V5 (001)
```

Each label `(s_a s_b s_c)` lists whether each top switch is on (`1`)
or off (`0`).


In [ ]:
# Plot the 6 active vectors in the αβ plane
def state_to_vector(s_a, s_b, s_c, V_dc=400.0):
    '''Map switching state (0/1 per leg) to αβ vector via Clarke.'''
    # Top switch on → that leg's pole at V_dc; otherwise at 0
    # Subtract bus midpoint for symmetry → v_pole_x = V_dc/2 or -V_dc/2
    v_a = V_dc * (s_a - 0.5)
    v_b = V_dc * (s_b - 0.5)
    v_c = V_dc * (s_c - 0.5)
    # Use clarke_transform (returns peak)
    arr_a = np.array([v_a]); arr_b = np.array([v_b]); arr_c = np.array([v_c])
    v_alpha, v_beta = clarke_transform(arr_a, arr_b, arr_c)
    return float(v_alpha[0]), float(v_beta[0])


states = [(1,0,0), (1,1,0), (0,1,0), (0,1,1), (0,0,1), (1,0,1), (0,0,0), (1,1,1)]
labels = ["V1 (100)", "V2 (110)", "V3 (010)", "V4 (011)",
          "V5 (001)", "V6 (101)", "V0 (000)", "V7 (111)"]

fig, ax = plt.subplots(figsize=(6, 6))
for st, lbl in zip(states, labels):
    va, vb = state_to_vector(*st, p.V_dc)
    if lbl.startswith("V0") or lbl.startswith("V7"):
        ax.plot(va, vb, "ko", markersize=10)
        ax.annotate(lbl, (va, vb), textcoords="offset points", xytext=(10, 5))
    else:
        ax.plot([0, va], [0, vb], "C0-")
        ax.plot(va, vb, "C0o", markersize=10)
        ax.annotate(lbl, (va, vb), textcoords="offset points", xytext=(8, 5))

# Plot the rotating reference circle (linear-mod limit for SVPWM)
theta_circ = np.linspace(0, 2*np.pi, 200)
R_limit = p.V_dc / np.sqrt(3.0)  # SVPWM linear-mod hexagon-inscribed circle
ax.plot(R_limit*np.cos(theta_circ), R_limit*np.sin(theta_circ), "C3--",
        alpha=0.5, label=f"Linear SVPWM limit: $V_{{dc}}/\\sqrt{{3}}$ = {R_limit:.0f} V")
R_spwm = p.V_dc / 2.0  # SPWM linear-mod limit
ax.plot(R_spwm*np.cos(theta_circ), R_spwm*np.sin(theta_circ), "C2:",
        alpha=0.5, label=f"SPWM limit: $V_{{dc}}/2$ = {R_spwm:.0f} V")
ax.axhline(0, color="k", linestyle=":", alpha=0.3)
ax.axvline(0, color="k", linestyle=":", alpha=0.3)
ax.set_xlabel("$v_\\alpha$ [V]"); ax.set_ylabel("$v_\\beta$ [V]")
ax.set_title("3-phase VSI: 6 active + 2 zero space vectors")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout(); plt.show()

print(f"SVPWM-vs-SPWM bonus: {R_limit/R_spwm:.4f}× = {(R_limit/R_spwm - 1)*100:.1f}% extra output")


## 2. SPWM — three independent SPWMs

The simplest modulation: generate three sinusoidal references
$v_{ref,a/b/c}(t)$ 120° apart, compare each to a common triangle
carrier at $f_{sw}$, and produce the three leg duties.

Per-leg duty:
$$d_x = 0.5 + v_{ref,x} / V_{dc}$$

with $|v_{ref}| \le V_{dc}/2$ for linear operation. The peak
**line-to-line** output is then bounded by $V_{dc} \cdot \sqrt 3 / 2
\approx 0.866 V_{dc}$ — that's the SPWM ceiling.


In [ ]:
# Generate one cycle of SPWM duties
t = np.linspace(0, 1.0/p.f_o, 1000)
V_ref_pk = p.m_a * p.V_dc / 2.0
v_ref_a = V_ref_pk * np.cos(p.omega_o * t)
v_ref_b = V_ref_pk * np.cos(p.omega_o * t - 2*np.pi/3)
v_ref_c = V_ref_pk * np.cos(p.omega_o * t + 2*np.pi/3)
d_a, d_b, d_c = spwm_duties(v_ref_a, v_ref_b, v_ref_c, p.V_dc)

fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axs[0].plot(t*1000, v_ref_a, label="$v_{ref,a}$")
axs[0].plot(t*1000, v_ref_b, "--", label="$v_{ref,b}$")
axs[0].plot(t*1000, v_ref_c, ":", label="$v_{ref,c}$")
axs[0].set_ylabel("Reference [V]"); axs[0].legend(loc="lower right")
axs[0].set_title("SPWM: three independent reference sinusoids → three duties")

axs[1].plot(t*1000, d_a, label="$d_a$")
axs[1].plot(t*1000, d_b, "--", label="$d_b$")
axs[1].plot(t*1000, d_c, ":", label="$d_c$")
axs[1].axhline(0.5, color="k", linestyle=":", alpha=0.4)
axs[1].set_ylabel("Duty cycle"); axs[1].set_xlabel("Time [ms]")
axs[1].legend(loc="lower right")
plt.tight_layout(); plt.show()
print(f"Duty range: [{d_a.min():.4f}, {d_a.max():.4f}]")


## 3. Open-loop SPWM switched simulation

Now drive the actual switching engine with SPWM. The 3-phase output
is filtered through an LC ($L_f = 1$ mH, $C_f = 10$ µF, per phase)
that strips the switching ripple and leaves a clean sinusoid.

We'll set `use_svpwm=False` to use standard SPWM. The next notebook
upgrades to SVPWM for the 15% bonus.


In [ ]:
sim = simulate_open_loop_svpwm(p, m_a=p.m_a, n_cycles=3, samples_per_period=80,
                                use_svpwm=False)
mask = sim['t'] > 1.0/p.f_o  # skip first cycle
fs = 1.0/(sim['t'][1] - sim['t'][0])
v_Ca_fund_rms = fundamental_rms(sim['v_Ca'][mask], fs, p.f_o)
v_Ca_thd = thd(sim['v_Ca'][mask], fs, p.f_o, 40)

fig, axs = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
axs[0].plot(sim['t']*1000, sim['v_pole_a'], "C0", linewidth=0.5, alpha=0.7,
            label="$v_{pole,a}$ (PWM)")
axs[0].set_ylabel("Pole voltage [V]"); axs[0].legend(loc="upper right")
axs[0].set_title(f"Open-loop SPWM, $m_a$ = {p.m_a:.3f}")

axs[1].plot(sim['t']*1000, sim['v_LL_ab'], "C3", linewidth=0.5, alpha=0.5,
            label="$v_{LL,ab}$ (pre-filter)")
axs[1].set_ylabel("Line-to-line [V]"); axs[1].legend(loc="upper right")

axs[2].plot(sim['t']*1000, sim['v_Ca'], "C2", linewidth=1.0, label="$v_{Ca}$")
axs[2].plot(sim['t']*1000, sim['v_Cb'], "C0", linewidth=1.0, label="$v_{Cb}$")
axs[2].plot(sim['t']*1000, sim['v_Cc'], "C1", linewidth=1.0, label="$v_{Cc}$")
axs[2].set_ylabel("Filtered phase [V]"); axs[2].set_xlabel("Time [ms]")
axs[2].legend(loc="lower right")
plt.tight_layout(); plt.show()

print(f"v_Ca fundamental rms = {v_Ca_fund_rms:.2f} V  (expect {p.V_o_LN_pk/np.sqrt(2):.2f} V)")
print(f"v_Ca THD             = {v_Ca_thd*100:.2f} %  (expect < 2 % with a clean filter)")


## 4. Clarke transform — three phases to αβ

Three real-valued sinusoids 120° apart can be encoded as a single
rotating 2D vector in the **αβ plane**. The Clarke transform makes
the encoding explicit:

$$
\begin{bmatrix} v_\alpha \\ v_\beta \end{bmatrix}
= \frac{2}{3}
\begin{bmatrix}
1 & -\tfrac{1}{2} & -\tfrac{1}{2} \\
0 & \tfrac{\sqrt 3}{2} & -\tfrac{\sqrt 3}{2}
\end{bmatrix}
\begin{bmatrix} v_a \\ v_b \\ v_c \end{bmatrix}
$$

The $2/3$ factor is the **amplitude-invariant** convention: with
$2/3$, the αβ vector magnitude equals the *peak* of the original
phase voltage. (Some textbooks use $\sqrt{2/3}$ for the
**power-invariant** version. We use $2/3$.)

For a balanced positive-sequence input
$v_a = V_{pk}\cos(\omega t)$, $v_b = V_{pk}\cos(\omega t - 2\pi/3)$,
$v_c = V_{pk}\cos(\omega t + 2\pi/3)$:

$$v_\alpha(t) = V_{pk} \cos(\omega t), \qquad v_\beta(t) = V_{pk} \sin(\omega t)$$

i.e. αβ traces a circle of radius $V_{pk}$ at angular frequency
$\omega$.


In [ ]:
# Verify on the open-loop sim output
v_alpha_sim, v_beta_sim = clarke_transform(sim['v_Ca'], sim['v_Cb'], sim['v_Cc'])

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
axs[0].plot(sim['t']*1000, v_alpha_sim, "C0", label="$v_\\alpha$")
axs[0].plot(sim['t']*1000, v_beta_sim, "C1", label="$v_\\beta$")
axs[0].set_xlabel("Time [ms]"); axs[0].set_ylabel("Voltage [V]")
axs[0].set_title("αβ components of the filter output")
axs[0].legend()

# αβ trajectory (the circle)
mask = sim['t'] > 1.0/p.f_o
axs[1].plot(v_alpha_sim[mask], v_beta_sim[mask], "C2", alpha=0.6)
axs[1].set_xlabel("$v_\\alpha$ [V]"); axs[1].set_ylabel("$v_\\beta$ [V]")
axs[1].set_title("αβ trajectory — a circle (positive sequence)")
axs[1].set_aspect("equal")
axs[1].axhline(0, color="k", linestyle=":", alpha=0.3)
axs[1].axvline(0, color="k", linestyle=":", alpha=0.3)
plt.tight_layout(); plt.show()
print(f"αβ peak: |v| = {np.sqrt(v_alpha_sim[mask]**2 + v_beta_sim[mask]**2).mean():.2f} V "
      f"(expect ≈ {p.V_o_LN_pk:.2f} V)")


## 5. Park transform — αβ to dq (the headline insight)

The αβ vector rotates at $\omega$. **Rotate the frame WITH IT** at
the same angular velocity, and the vector becomes stationary —
i.e. a **DC** quantity. That's the Park transform:

$$
\begin{bmatrix} v_d \\ v_q \end{bmatrix}
=
\begin{bmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{bmatrix}
\begin{bmatrix} v_\alpha \\ v_\beta \end{bmatrix},
\quad \theta = \omega t
$$

For positive-sequence input with d-axis aligned to phase-a at
$t = 0$:

$$v_d(t) = V_{pk}, \qquad v_q(t) = 0$$

**Two constants** instead of three sinusoids. The compensator now
acts on DC quantities, with all the familiar tooling: PI for steady-
state, plant looks first-order or second-order in dq.

This is *the* trick that makes 3-phase control tractable.


In [ ]:
# Build the Park angle θ = ω_o · t and transform
theta = p.omega_o * sim['t']
v_d_sim, v_q_sim = park_transform(v_alpha_sim, v_beta_sim, theta)

fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axs[0].plot(sim['t']*1000, v_alpha_sim, "C0", alpha=0.5, label="$v_\\alpha$")
axs[0].plot(sim['t']*1000, v_beta_sim, "C1", alpha=0.5, label="$v_\\beta$")
axs[0].set_ylabel("αβ [V]"); axs[0].legend(loc="lower right")
axs[0].set_title("Three sinusoids → αβ (two sinusoids) → dq (two DC quantities)")

axs[1].plot(sim['t']*1000, v_d_sim, "C2", linewidth=1.5, label="$v_d$")
axs[1].plot(sim['t']*1000, v_q_sim, "C3", linewidth=1.5, label="$v_q$")
axs[1].axhline(p.V_o_LN_pk, color="k", linestyle=":", alpha=0.4,
               label=f"$V_d$ expected = {p.V_o_LN_pk:.1f} V")
axs[1].axhline(0, color="k", linestyle=":", alpha=0.4)
axs[1].set_ylabel("dq [V]"); axs[1].set_xlabel("Time [ms]")
axs[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

mask = sim['t'] > 1.5/p.f_o
print(f"Steady-state v_d mean = {v_d_sim[mask].mean():.2f} V  (expect {p.V_o_LN_pk:.2f})")
print(f"Steady-state v_q mean = {v_q_sim[mask].mean():.4f} V  (expect 0)")
print(f"v_d ripple std = {v_d_sim[mask].std():.4f} V")
print(f"v_q ripple std = {v_q_sim[mask].std():.4f} V")


## 6. Summary

You've seen:

- The 3-phase VSI topology with its 8 switching states.
- How to generate SPWM duties from three references.
- The Clarke transform: 3 phases → 2 αβ components (a rotating vector
  in 2D).
- The Park transform: αβ → dq, mapping AC sinusoids to **DC**
  quantities by rotating with the line.

The dq frame is what enables straightforward control of a 3-phase
inverter — the buck-control intuition transfers verbatim.

**Next notebooks**:

- `02_vsi_svpwm.ipynb` — derive SVPWM, prove the min-max injection
  equivalence, show the 15% output bonus.
- `03_vsi_standalone.ipynb` — design the dq voltage compensator and
  run the closed-loop sim with an RL load.
- `04_vsi_gridtie.ipynb` — wire the SRF-PLL and the dq current
  control, inject active and reactive power.
